# SpatialMETA parameter benchmarking

## Imports

In [1]:
import seaborn as sns
import spatialmeta as smt
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import time
import psutil
import os

#from memory_profiler import memory_usage

## Loading data

In [2]:
joint_adata = smt.data.load_adata(
    sample_name="Y7_T_raw",
    modality="joint"
)

/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/site-packages/spatialmeta/data/./datasets/adata_joint_Y7_T_raw_raw.h5ad


In [3]:
type(joint_adata)

anndata._core.anndata.AnnData

## Preprocessing

In [4]:
process = psutil.Process(os.getpid())

mem_before = process.memory_info().rss / 1024**2
start = time.perf_counter()
####

joint_adata = smt.pp.removeHSP_MT_RPL_DNAJ(joint_adata)

joint_adata.layers["counts"] = joint_adata.X.copy()

smt.pp.normalize_total_joint_adata_sm_st(joint_adata,
                         target_sum_SM=1e4,
                         target_sum_ST=1e4)

joint_adata.layers["normalized"] = joint_adata.X.copy()

joint_adata.raw = joint_adata

smt.pp.spatial_variable_joint_adata_sm_st(joint_adata,
                                         n_top_genes = 2000,
                                         n_top_metabolites = 800,
                                         add_key = "highly_variable_moranI")

joint_adata = joint_adata[:,joint_adata.var.highly_variable_moranI]

joint_adata.write_h5ad("SpatialMETA/data/Y7_T_adata_joint_hvf2800.h5ad")

####
end = time.perf_counter()
mem_after = process.memory_info().rss / 1024**2

print(f"Runtime: {end-start:.2f} s")
print(f"Memory before: {mem_before:.2f} MB")
print(f"Memory after: {mem_after:.2f} MB")
print(f"Difference: {mem_after-mem_before:.2f} MB")

Runtime: 1.61 s
Memory before: 816.34 MB
Memory after: 1724.09 MB
Difference: 907.75 MB


## CVAE model

In [6]:
joint_adata = sc.read_h5ad("SpatialMETA/data/Y7_T_adata_joint_hvf2800.h5ad")

joint_adata.X = joint_adata.layers["counts"]

smt.pp.normalize_total_joint_adata_sm_st(
    joint_adata,
    target_sum_SM=1e3,
    target_sum_ST=None
)

model = smt.model.ConditionalVAESTSM(
    joint_adata,
    device='cpu', # Small change to CPU instead of CUDA
    reconstruction_method_sm='g',
    reconstruction_method_st='zinb',
)

loss_dict = model.fit(
    max_epoch=64,
    lr=1e-3,
    mode='single'
)

####
####

Z = model.get_latent_embedding()
X = model.get_normalized_expression()
C = model.get_modality_contribution()

joint_adata.layers['reconstruction'] = X
joint_adata.obsm['X_emb']=Z
joint_adata.obs['contribution_st']=C
joint_adata.obs['contribution_sm']=1-C

####
####

from multi_benmark_function import (
    compute_CHAOS, compute_PAS,
    marker_score,
    compute_ARI, compute_NMI,
    compute_ASW, compute_gt_silhouette,
    compute_clisi_graph, compute_isolated_aws,
    calculate_gene_specificity,
    logistic_regression_feature_importance,
    calculate_mutual_information
)
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

PRED_KEY = 'VAE_clusters_latent10'
EMB_KEY   = 'X_emb'
GT_KEY    = 'pathological_annotation'

####

import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity

# Pull out dense matrices
X_orig  = joint_adata.layers['counts']
X_recon = joint_adata.layers['reconstruction']

if hasattr(X_orig, 'toarray'):
    X_orig = X_orig.toarray()
if hasattr(X_recon, 'toarray'):
    X_recon = X_recon.toarray()

# PCC — one value per spot, then averaged
pccs = [pearsonr(X_orig[i], X_recon[i])[0] for i in range(X_orig.shape[0])]
mean_pcc = np.nanmean(pccs)

# Cosine Similarity — diagonal of the pairwise matrix gives per-spot values
cs_matrix = cosine_similarity(X_orig, X_recon)
mean_cs = np.nanmean(cs_matrix.diagonal())

print(f"Mean PCC: {mean_pcc:.4f}")
print(f"Mean Cosine Similarity: {mean_cs:.4f}")

####

chaos_score = compute_CHAOS(
    joint_adata,
    pred_key='VAE_clusters_latent10',
    spatial_key='spatial'         # adata.obsm['spatial']
)
print(f"CHAOS: {chaos_score:.4f}")

####

pas_score = compute_PAS(
    joint_adata,
    pred_key='VAE_clusters_latent10',
    spatial_key='spatial'
)
print(f"PAS: {pas_score:.4f}")

####

# Prep: set X to counts and make sure leiden key matches
adata_ms = joint_adata.copy()
adata_ms.X = adata_ms.layers['counts'].copy()
adata_ms.obs[PRED_KEY] = adata_ms.obs[PRED_KEY].astype('category')

# ST modality
adata_ST = adata_ms[:, adata_ms.var['type'] == 'ST'].copy()
moranI_ST, gearyC_ST = marker_score(adata_ST, domain_key=PRED_KEY, top_n=50)
print(f"ST  Moran's I: {moranI_ST:.4f}  |  Geary's C: {gearyC_ST:.4f}")

# SM modality
adata_SM = adata_ms[:, adata_ms.var['type'] == 'SM'].copy()
moranI_SM, gearyC_SM = marker_score(adata_SM, domain_key=PRED_KEY, top_n=50)
print(f"SM  Moran's I: {moranI_SM:.4f}  |  Geary's C: {gearyC_SM:.4f}")

# Moran's I: higher (→1) = stronger spatial structure
# Geary's C: lower (→0) = stronger spatial structure

####

specificity_SM = calculate_gene_specificity(adata_SM, domain_key=PRED_KEY, layers="counts")
specificity_ST = calculate_gene_specificity(adata_ST, domain_key=PRED_KEY, layers="counts")

print(f"SM  specificity: {specificity_SM:.4f}  |  ST specificity: {specificity_ST:.4f}")

####

logistic_ST = logistic_regression_feature_importance(adata_ST, domain_key=PRED_KEY, layers="counts")
logistic_SM = logistic_regression_feature_importance(adata_SM, domain_key=PRED_KEY, layers="counts")

print(f"SM  feature importance: {logistic_SM:.4f}  |  ST feature importance: {logistic_ST:.4f}")

####

mi_ST = calculate_mutual_information(adata_SM, domain_key=PRED_KEY, layers="counts")
mi_SM = calculate_mutual_information(adata_ST, domain_key=PRED_KEY, layers="counts")

print(f"SM  mutual information: {logistic_SM:.4f}  |  ST mutual information: {logistic_ST:.4f}")

####

ari = compute_ARI(joint_adata, gt_key=PRED_KEY, pred_key=PRED_KEY)
nmi = compute_NMI(joint_adata, gt_key=PRED_KEY, pred_key=PRED_KEY)
print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")
# Both range 0–1, higher is better

####

# Spatial ASW — silhouette in physical space w.r.t. predicted clusters
asw_spatial = compute_ASW(
    joint_adata,
    pred_key=PRED_KEY,
    spatial_key='spatial'
)
print(f"Spatial ASW: {asw_spatial:.4f}")

# Embedding ASW — silhouette in X_emb space w.r.t. ground truth (scib version)
# Requires GT_KEY in adata.obs
asw_emb = compute_gt_silhouette(
    joint_adata,
    gt_key=PRED_KEY,
    embedding_key=EMB_KEY
)
print(f"Embedding ASW (gt): {asw_emb:.4f}")
# Range -1 to 1, higher is better

####

clisi = compute_clisi_graph(
    joint_adata,
    gt_key=PRED_KEY,
    embedding_key=EMB_KEY
)
print(f"cLISI: {clisi:.4f}")
# Higher = cell-type neighborhoods are purer in embedding space

####

isolated_asw = compute_isolated_aws(
    joint_adata,
    gt_key=PRED_KEY,
    embedding_key=EMB_KEY
)
print(f"Isolated label ASW: {isolated_asw:.4f}")
# Higher = rare cell types are well-separated in embedding space

####

import pandas as pd

results = {
    'PCC':           mean_pcc,
    'CosineSim':     mean_cs,
    'CHAOS':         chaos_score,
    'PAS':           pas_score,
    'ST_MoranI':     moranI_ST,
    'ST_GearyC':     gearyC_ST,
    'SM_MoranI':     moranI_SM,
    'SM_GearyC':     gearyC_SM,
    'ST_Specificity': specificity_ST,
    'SM_Specificity': specificity_SM,
    'ST_Logistic': logistic_ST,
    'SM_Logistic': logistic_SM,
    'ST_Mutual Information': mi_ST,
    'SM_Mutual Information': mi_SM,
    'ARI':           ari,
    'NMI':           nmi,
    'ASW_spatial':   asw_spatial,
    'ASW_emb':       asw_emb,
    'cLISI':         clisi,
    'Isolated_ASW':  isolated_asw,
}

reconstrucion_accuracy = (mean_pcc + mean_cs)/2
continuity_score = ((1 - chaos_score) + (1 - pas_score))/2
# MoransI + 1-Geary'sG + Specificity + Logistic + MI / 5
marker_score_ST = (moranI_ST + (1 - gearyC_ST) + specificity_ST + mi_ST)/5
marker_score_SM = (moranI_SM + (1 - gearyC_SM) + specificity_SM + mi_SM)/5
marker_score = (moranI_ST + (1 - gearyC_ST) + specificity_ST + mi_ST + moranI_SM + (1 - gearyC_SM) + specificity_SM + mi_SM) / 10
biological_conservation = (ari + nmi + asw_spatial + isolated_asw + clisi)/5

results_metrics = {"Continuity Score": continuity_score,
                   "Marker score": marker_score,
                   "Marker Score ST": marker_score_ST,
                   "Marker score SM": marker_score_SM,
                   "Biological Conservation": biological_conservation,
                   "Reconstruction Accuracy": reconstrucion_accuracy
                   }

results = pd.DataFrame(results, index=['spatialMETA'])

results_metrics = pd.DataFrame(results_metrics, index=['spatialMETA'])

Latent Embedding: 100%|██████████| 15/15 [00:00<00:00, 118.27it/s]
2026-06-03 10:02:59,516 - WARNING - compute_CHAOS (adata): Key VAE_clusters_latent10 or spatial not in adata.obs/obsm. Skipping.
2026-06-03 10:02:59,517 - WARNING - compute_PAS (adata): Key VAE_clusters_latent10 or spatial not in adata.obs/obsm. Skipping.


Mean PCC: 0.5958
Mean Cosine Similarity: 0.6191
CHAOS: nan
PAS: nan


KeyError: 'VAE_clusters_latent10'